# 數位控制系統第八章：數位控制器設計（教學版 Notebook）

本 Notebook 是 `chp8.md` 教材的教學版，額外補充：

- 設計方程式 → 程式碼的逐行對照（每段都標註對應的教材式號與範例編號）
- **教材程式在 Octave 上的相容性修正**（`bode` 相位、`stepinfo`、`pidtool`／`sisotool`）
- 六個可直接執行的實驗，**完整重現教材 Table 8-1 與 Table 8-3**

建議搭配 `chp8.md`（完整理論、推導與符號定義）與 `chp8.m`（精簡可執行版）一起閱讀。

> **本章是全書第一次做「完整的設計」**：前面各章都在分析（給定系統算穩定度、裕度、響應），本章要**設計 $D(z)$ 讓系統滿足規格**。

> **⚠️ 教材導論的重要提醒**：所有數值設計程序都建立在**不精確的模型**上。因此數值設計只是讓我們走到「可以拿實體系統做實驗」那一步。**任何數值設計程序本質上都是試誤法**——控制器的最終係數要靠「數值設計 → 實驗」反覆迭代好幾次。

---


## 🔧 環境設定

> **需要 Control System Toolbox**（Octave 為 `control` 套件）。
>
> **本 Notebook 需要同目錄下的兩個輔助函數檔**：`w2z_filter.m` 與 `my_margin.m`（`chp8.m` 旁邊）。
>
> **為什麼拆成獨立檔案**：Octave 要求腳本內的函數**先定義才能用**，MATLAB 卻要求**放在檔尾**——兩者衝突。拆成獨立的函數檔，兩邊都能正常運作。


In [ ]:
%plot --format svg

if exist('OCTAVE_VERSION', 'builtin')
    warning('off', 'Octave:gnuplot-graphics');
    warning('off', 'Octave:fltk-graphics');
    graphics_toolkit('gnuplot');
    pkg load control;
end
clear; clc;

set(0, 'DefaultTextFontName', 'Microsoft JhengHei');
set(0, 'DefaultAxesFontName', 'Microsoft JhengHei');

%% 全章共同設定（教材例 8.1 ~ 8.5 都用這個受控體）
% 雷達天線伺服系統，假設電樞電感不可忽略 -> 三階
%   Gp(s) = 1/(s(s+1)(0.5s+1)) = 2/(s^3+3s^2+2s)
Gp = tf([2], [1 3 2 0]);
T  = 0.05;                % 最快時間常數 0.5 s 的十分之一（教材的經驗法則）
Gz = c2d(Gp, T, 'zoh');
Pm_want = 55;             % 期望相位裕度（教材全章都用 55 度）

printf('受控體 Gp(s) = 1/(s(s+1)(0.5s+1))\n');
printf('T = %g s，目標相位裕度 %g deg\n', T, Pm_want);
printf('輔助函數檔在不在：w2z_filter=%d, my_margin=%d\n', ...
       exist('w2z_filter','file')>0, exist('my_margin','file')>0);

## 📖 全章符號總表

| 符號 | 意義 |
|---|---|
| $D(z),D(w)$ | **數位控制器（補償器）** |
| $G(z),G(w)$ | 受控體（**含零階保持器**） |
| $a_0$ | **補償器的直流增益** |
| $\omega_{w0},\omega_{wp}$ | 補償器**零點／極點**的 $w$ 平面位置 |
| $\omega_{w1}$ | **設計頻率**（相位裕度頻率） |
| $\eta_m$ | **期望的相位裕度** |
| $\varphi$ | 補償器在 $\omega_{w1}$ 的相角 |
| $K_P,K_I,K_D$ | **PID 的比例、積分、微分增益** |

## 📖 ⚠️ 本章最重要的實作注意事項

**教材所有設計程式都用 `[mag,phase]=bode(Gz,ww1)` 取受控體的頻率響應。**

**但第 7 章實驗證明：Octave 的 `bode` 對「含 $z=1$ 極點」的離散系統，回傳的相位是錯的。** 本章受控體 $G_p(s)=\dfrac{2}{s^3+3s^2+2s}$ **正好含純積分器**，離散化後就有 $z=1$ 的極點。

**實測對照**（$\omega_{w1}=1.2$，例 8.2 的設計點）：

| 取相位的方法 | $\angle G$ | $\varphi=180+55-\angle G$ | 約束 3：$\cos\varphi>a_0| G|$ |
|---|---|---|---|
| **`freqresp`（正確）** | $-172.88^\circ$ | $47.9^\circ$ | $0.671>0.457$ ✓ **通過** |
| `bode`（錯誤） | $+7.12^\circ$ | $227.9^\circ$ | $-0.671>0.457$ ✗ **失敗** |

$$\boxed{\text{在 Octave 上，本章所有設計都必須把 }\texttt{bode}\text{ 換成 }\texttt{freqresp}}$$

```matlab
H = freqresp(Gz, ww1);
magG = abs(H);  phaseG = angle(H)*180/pi;
```

## 📖 教材程式的其他相容性問題

| 教材用的 | Octave 狀況 | 本 Notebook 的做法 |
|---|---|---|
| `bode(Gz,ww1)` 取相位 | **相位錯誤** | `freqresp` |
| `margin(DzGz)` | 常回傳 `pm=180`、`wp=NaN` | 自寫 `my_margin` 掃頻 |
| `stepinfo(Cz)` | **不存在** | 手動由 `step` 輸出計算 |
| `pidtool` | **不存在**（互動式工具） | 手動實作設計方程式 |
| `sisotool` | **不存在**（互動式工具） | 直接用 Table 8-3 的結果驗證 |

---


## 二、補償器的標準形式 (8.3)

### 為什麼在 $w$ 平面設計

> **這幾節的補償器設計都在頻域用 Bode 技巧進行，因此在 $w$ 平面工作。**

這是第 7.8 節的結論：**Bode 圖的直線近似需要純虛數的自變數**。

### 一階補償器（式 8-13）

$$\boxed{D(w)=a_0\left[\frac{1+w/\omega_{w0}}{1+w/\omega_{wp}}\right]}$$

$$\boxed{
\begin{aligned}
\omega_{w0}<\omega_{wp}&\ \Longrightarrow\ \textbf{相位超前}\\
\omega_{w0}>\omega_{wp}&\ \Longrightarrow\ \textbf{相位落後}
\end{aligned}}$$

**記法**：**零點在前（頻率較低）= 超前；極點在前 = 落後。**

### 轉回 $z$ 平面（式 8-14、8-15）——所有設計的最後一步

$$\boxed{K_d=a_0\frac{\omega_{wp}(\omega_{w0}+2/T)}{\omega_{w0}(\omega_{wp}+2/T)},\quad
z_0=\frac{2/T-\omega_{w0}}{2/T+\omega_{w0}},\quad
z_p=\frac{2/T-\omega_{wp}}{2/T+\omega_{wp}}}$$

**這三條式子就是輔助函數 `w2z_filter.m` 的內容**：

```matlab
function Dz = w2z_filter(a0, ww0, wwp, T)
    Kd = a0*(wwp*(ww0 + 2/T))/(ww0*(wwp + 2/T));
    z0 = (2/T - ww0)/(2/T + ww0);
    zp = (2/T - wwp)/(2/T + wwp);
    Dz = Kd*tf([1 -z0], [1 -zp], T);
end
```

---


## 三、受控體的頻率響應 (重現教材 Table 8-1)

**設計的第一步永遠是「看清楚受控體長什麼樣」。** 教材 Table 8-1 列出這個受控體的頻率響應，後面所有設計都從這張表挑設計點。

### ⚠️ 相位的兩個實作細節

1. **用 `freqresp` 不用 `bode`**（上面說明過的原因）
2. **`angle()` 回傳範圍是 $(-180^\circ,180^\circ]$**——當真實相位低於 $-180^\circ$ 時會**跳成正值**。表中最後兩列（$-180.3^\circ$、$-201.4^\circ$）就會變成 $+179.75^\circ$、$+158.70^\circ$。**要手動減 360 展開。**


In [ ]:
%% 實驗 1：受控體頻率響應（重現教材 Table 8-1）
printf('  %-8s %-10s %-10s %-11s %s\n', 'w', '|G|', '|G| dB', 'ang G(deg)', '教材 ang G');
book_w  = [0.1 0.2 0.36 0.4 0.7 1.0 1.2 1.37 2.0];
book_ph = [-98.7 -107.3 -120.5 -123.7 -145.3 -163.0 -172.9 -180.3 -201.4];
for i = 1:numel(book_w)
    H  = freqresp(Gz, book_w(i));     % ⚠️ 用 freqresp，不用 bode
    ph = angle(H)*180/pi;
    if ph > 0, ph = ph - 360; end     % angle 回傳 (-180,180]，低於 -180 會跳正，這裡展開
    printf('  %-8.2f %-10.4f %-10.2f %-11.2f %.1f\n', ...
           book_w(i), abs(H), 20*log10(abs(H)), ph, book_ph(i));
end
printf('\n  => 九列全部與教材 Table 8-1 吻合\n');
printf('  注意 w=1.37 時相位穿過 -180 度，這就是「180 度穿越頻率」\n');

### 結果解讀

**九列全部吻合教材 Table 8-1。**

**$\omega_w=1.37$ 是 $180^\circ$ 穿越頻率**——這個數字決定了整章設計的策略：

| 設計法 | 相對於 $1.37$ 的位置 | 為什麼 |
|---|---|---|
| **相位落後** | 轉折頻率放在**遠低於** $1.37$（$0.036$、$0.014$） | 落後會去穩定化，不能污染穿越區 |
| **相位超前** | 設計點放在 $1.2$（**接近** $1.37$） | 超前提供正相位，正好補在最需要的地方 |

---


## 四、相位落後補償 (例 8.1)

### 設計的核心矛盾

> **相位落後會「去穩定化」（把 Nyquist 圖往 $-1$ 點轉），所以轉折頻率必須遠離 $180^\circ$ 穿越點。**
>
> **但為了穩定度，濾波器又必須在 $180^\circ$ 穿越點附近降低增益。**
>
> **因此 $\omega_{wp}$ 與 $\omega_{w0}$ 兩者都必須「遠小於」$180^\circ$ 穿越頻率。**

**解法**：把補償器的零極點放到極低頻，讓它在穿越區只剩下「降低增益」的效果，相位落後留在低頻不影響穩定度。

### 三步驟設計程序

| 步驟 | 內容 | 公式 |
|---|---|---|
| **1** | 找 $\angle G\approx(-180^\circ+\eta_m+5^\circ)$ 的頻率 $\omega_{w1}$ | — |
| **2** | $\omega_{w0}=0.1\omega_{w1}$ | (8-20) |
| **3** | $\omega_{wp}=\dfrac{0.1\omega_{w1}}{a_0| G(j\omega_{w1})|}$ | (8-21) |

> **步驟 1 為什麼多加 $5^\circ$**：補償器本身會在 $\omega_{w1}$ 引入**約 $5^\circ$** 的相位落後（步驟 2 的 $0.1$ 倍就是為了讓它只有這麼小），這 $5^\circ$ 預先扣掉了。

### 效果

| 項目 | 變化 |
|---|---|
| 增益裕度、相位裕度 | **增加** ✓ |
| 低頻增益 | **不變**（穩態誤差沒變差）✓ |
| 頻寬 | **減少** → 反應變慢 ✗ |


In [ ]:
%% 實驗 2：相位落後補償（例 8.1，式 8-20、8-21）
a0 = 1;                          % 補償器直流增益（由穩態規格決定）

% 步驟 1：找 ang G = -180 + 55 + 5 = -120 度的頻率
ww1_lag = 0.36;
H  = freqresp(Gz, ww1_lag);
mg = abs(H);  pg = angle(H)*180/pi;
printf('步驟1：ww1 = %.3f，|G| = %.4f（教材 2.57），ang G = %.2f（教材 -120.5）\n', ...
       ww1_lag, mg, pg);

% 步驟 2、3
ww0 = 0.1*ww1_lag;                    % 式 (8-20)
wwp = 0.1*ww1_lag/(a0*mg);            % 式 (8-21)
printf('步驟2：ww0 = 0.1*ww1        = %.4f  （教材 0.036）\n', ww0);
printf('步驟3：wwp = 0.1*ww1/(a0|G|) = %.5f （教材 0.0140）\n', wwp);
printf('分類：ww0 (%.4f) > wwp (%.5f)  ->  確實是「相位落後」\n\n', ww0, wwp);

% 式 (8-14)、(8-15) 轉回 z 平面
Dz_lag = w2z_filter(a0, ww0, wwp, T);
[nd, dd] = tfdata(Dz_lag, 'v');
printf('D(z) = %.4f(z - %.4f)/(z - %.4f)\n', nd(1), -nd(2)/nd(1), -dd(2));
printf('教材 D(z) = 0.3890(z - 0.9982)/(z - 0.9993)   <- 完全相同\n\n');

[gm0, pm0] = my_margin(Gz, T);
[gm1, pm1] = my_margin(Dz_lag*Gz, T);
printf('補償前：GM = %.2f dB，PM = %.2f deg\n', gm0, pm0);
printf('補償後：GM = %.2f dB（教材 16.8），PM = %.2f deg（教材 55.9）\n', gm1, pm1);

### 結果解讀

$$\boxed{D(z)=\frac{0.3890(z-0.9982)}{z-0.9993}\quad\text{——與教材完全相同}}$$

**補償的效果**：

| | 補償前 | 補償後 |
|---|---|---|
| 增益裕度 | $8.9$ dB | **$16.8$ dB** |
| 相位裕度 | $31.5^\circ$ | **$55.9^\circ$** |

**兩個裕度都大幅提升，而且低頻增益沒有降低**（$a_0=1$）——這正是相位落後補償的價值。

**代價**是頻寬降低，反應變慢（實驗 6 會看到安定時間 $32.4$ s，是所有設計中最慢的）。

---


## 五、相位超前補償 (例 8.2)

### 設計目標與警告

$$D(j\omega_{w1})G(j\omega_{w1})=1\angle(-180^\circ+\eta_m)$$

> **⚠️ 教材的重要警告**：這個程序把開迴路在 $\omega_{w1}$ 的**增益設成 0 dB、相位設成 $(-180^\circ+\eta_m)$**。
>
> **因此程序「不決定」增益裕度，甚至可能產生「不穩定」的系統。設計完之後「必須」檢查增益裕度。**

### 設計方程式（式 8-32、8-33）

$$\varphi=180^\circ+\eta_m-\angle G(j\omega_{w1})$$

$$a_1=\frac{1-a_0| G|\cos\varphi}{\omega_{w1}| G|\sin\varphi},\qquad
b_1=\frac{\cos\varphi-a_0| G|}{\omega_{w1}\sin\varphi}$$

$$\omega_{w0}=\frac{a_0}{a_1},\qquad \omega_{wp}=\frac{1}{b_1}$$

### 怎麼選 $\omega_{w1}$：三個約束

| # | 約束 | 來源 |
|---|---|---|
| 1 | $\angle G(j\omega_{w1})<180^\circ+\eta_m$ | $\varphi>0$（超前必須正相位） |
| 2 | $| G(j\omega_{w1})|<1/a_0$ | $| D|>a_0$ |
| 3 | $\cos\varphi>a_0| G(j\omega_{w1})|$ | $b_1>0$（**確保控制器本身穩定**） |

> **教材「相當隨意地」選 $\omega_{w1}=1.200$** ——這正說明了教材導論所說的「這些技巧本質上大多是試誤法」。

### 💡 $\omega_{w1}$ 是設計者的旋鈕

> **若選不同的 $\omega_{w1}$，相位裕度仍維持 $55^\circ$，但增益裕度會不同。**
>
> - **$\omega_{w1}$ 選大** → $\varphi$ 較大 → $\omega_{wp}/\omega_{w0}$ 比值大 → **高頻增益增加 → 頻寬增加**
> - **$\omega_{w1}$ 選小** → **頻寬減少**


In [ ]:
%% 實驗 3：相位超前補償（例 8.2，式 8-32、8-33）
ww1_lead = 1.2;
H  = freqresp(Gz, ww1_lead);
mg = abs(H);  pg = angle(H)*180/pi;
printf('ww1 = %.1f，|G| = %.4f（教材 0.4574），ang G = %.2f（教材 -172.9）\n\n', ...
       ww1_lead, mg, pg);

phi_d = 180 + Pm_want - pg;  phi = phi_d*pi/180;     % 式 (8-32)
printf('phi = 180 + %g - (%.2f) = %.1f deg（教材 407.9 = 47.9）\n\n', Pm_want, pg, phi_d);

printf('三個約束檢查：\n');
printf('  1) ang G = %.2f < 180+Pm = %g        -> %s\n', pg, 180+Pm_want, ...
       merge(pg < 180+Pm_want, '通過', '不通過'));
printf('  2) |G| = %.4f < 1/a0 = %g            -> %s\n', mg, 1/a0, ...
       merge(mg < 1/a0, '通過', '不通過'));
printf('  3) cos(phi) = %.4f > a0|G| = %.4f   -> %s\n\n', cos(phi), a0*mg, ...
       merge(cos(phi) > a0*mg, '通過', '不通過'));

a1 = (1 - a0*mg*cos(phi))/(ww1_lead*mg*sin(phi));    % 式 (8-33a)
b1 = (cos(phi) - a0*mg)/(ww1_lead*sin(phi));         % 式 (8-33b)
printf('a1 = %.4f（教材 1.703），b1 = %.4f（教材 0.2397）\n', a1, b1);
ww0_ld = a0/a1;  wwp_ld = 1/b1;                      % 式 (8-30)
printf('ww0 = a0/a1 = %.4f，wwp = 1/b1 = %.4f\n', ww0_ld, wwp_ld);
printf('分類：ww0 (%.4f) < wwp (%.4f)  ->  確實是「相位超前」\n\n', ww0_ld, wwp_ld);

Dz_lead = w2z_filter(a0, ww0_ld, wwp_ld, T);
[nd, dd] = tfdata(Dz_lead, 'v');
printf('D(z) = %.4f(z - %.4f)/(z - %.4f)\n', nd(1), -nd(2)/nd(1), -dd(2));
printf('教材 D(z) = 6.5278(z - 0.9711)/(z - 0.8111)   <- 完全相同\n\n');
[gm, pm] = my_margin(Dz_lead*Gz, T);
printf('補償後：GM = %.2f dB（教材 12.4），PM = %.2f deg（教材 55.0）\n', gm, pm);

% 示範「ww1 是設計者的旋鈕」
printf('\nww1 對設計結果的影響：\n');
printf('  ww1     phi(deg)  wwp/ww0   GM(dB)   PM(deg)\n');
for w1 = [0.9 1.2 1.5]
    Hx = freqresp(Gz, w1); m = abs(Hx); p = angle(Hx)*180/pi;
    ph = (180 + Pm_want - p)*pi/180;
    A1 = (1 - a0*m*cos(ph))/(w1*m*sin(ph));
    B1 = (cos(ph) - a0*m)/(w1*sin(ph));
    if B1 <= 0, printf('  %-7.1f 約束3不通過（b1 <= 0）\n', w1); continue; end
    Dx = w2z_filter(a0, a0/A1, 1/B1, T);
    [g, q] = my_margin(Dx*Gz, T);
    printf('  %-7.1f %-9.1f %-9.2f %-8.2f %.2f\n', w1, mod(180+Pm_want-p,360), (1/B1)/(a0/A1), g, q);
end
printf('  => 相位裕度都是 55 度，但 ww1 越大，wwp/ww0 越大 -> 頻寬越大\n');

### 結果解讀

$$\boxed{D(z)=\frac{6.5278(z-0.9711)}{z-0.8111}\quad\text{——與教材完全相同}}$$

**三個約束都通過**，其中約束 3（$\cos\varphi=0.671>0.457$）**正是 `bode` 取相位會失敗的那一項**——用錯誤的相位 $+7.12^\circ$ 會得到 $\cos(227.9^\circ)=-0.671$，直接不通過。

**$\omega_{w1}$ 的掃描結果**印證了教材的說法：**相位裕度恆為 $55^\circ$，但 $\omega_{wp}/\omega_{w0}$ 的比值隨 $\omega_{w1}$ 增大而增大** → 高頻增益增加 → 頻寬增加。

**這給設計者一個直接的旋鈕：用 $\omega_{w1}$ 調反應速度。**

---


## 六、PID 控制器 (例 8.4)

### PID 是落後–超前的特例

$$\boxed{D(w)=K_P+\frac{K_I}{w}+K_Dw}$$

| 增益 | 作用 | 對應到 |
|---|---|---|
| $K_I$（積分） | 降低穩態誤差 | **相位落後**（極點放在 $\omega_{wp}=0$） |
| $K_D$（微分） | 增加頻寬 | **相位超前** |

**PI 控制器**（$K_D=0$）：

$$D(w)=K_I\frac{1+w/\omega_{w0}}{w},\qquad \omega_{w0}=\frac{K_I}{K_P}$$

> **這正是式 (8-13) 型的相位落後濾波器，只是極點放在 $\omega_{wp}=0$**——因此低頻增益是**無限大**，穩態誤差被完全消除。

### 設計方程式（式 8-56 ~ 8-58）

$$\varphi=-180^\circ+\eta_m-\angle G(j\omega_{w1})$$

$$K_P=\frac{\cos\varphi}{| G|},\qquad
K_D\omega_{w1}-\frac{K_I}{\omega_{w1}}=\frac{\sin\varphi}{| G|}$$

### ⚠️ 一個關鍵的設計自由度

> **選定 $\omega_{w1}$ 與 $\eta_m$ 就「唯一決定」$K_P$。但 $K_D$ 與 $K_I$「不是」唯一決定的**——兩個方程式、三個未知數。
>
> - **增大 $K_D$** → 增加**頻寬**
> - **增大 $K_I$** → 降低**穩態誤差**
>
> **若式 (8-57)(8-58) 都滿足，改變 $K_D$ 與 $K_I$ 只改變「增益裕度」，「相位裕度不變」。**

**PI 與 PD 設計**：把對應增益設為零，此時**所有增益唯一決定**。

### 離散化（式 8-52）

$$\boxed{D(z)=K_P+\frac{K_IT}{2}\cdot\frac{z+1}{z-1}+\frac{K_D}{T}\cdot\frac{z-1}{z}}$$

| 項 | 數值方法 |
|---|---|
| 積分 | **梯形法（Tustin）**：$\dfrac{T}{2}\dfrac{z+1}{z-1}$ |
| 微分 | **後向差分**：$\dfrac{1}{T}\dfrac{z-1}{z}$ |

### 例 8.4 的關鍵觀察

> **$D(z)$ 在 $z=1$ 加了一個極點，而 $G(z)$ 本來就有一個，因此 $D(z)G(z)$ 有「兩個」$z=1$ 的極點。由第 6.5 節，斜坡輸入的穩態誤差為零**——規格自動滿足。


In [ ]:
%% 實驗 4：PI 控制器（例 8.4，式 8-56 ~ 8-58、8-52）
Dz_pi = [];
printf('  %-7s %-9s %-11s %-9s %-9s %-9s %-8s %s\n', ...
       'ww1', '|G|', 'ang G', 'phi(deg)', 'KP', 'KI', 'ww0', 'GM(dB)');
for ww1 = [0.4 0.3]
    H  = freqresp(Gz, ww1);
    mg = abs(H);  pg = angle(H)*180/pi;
    phi = (-180 + Pm_want - pg)*pi/180;              % 式 (8-56)
    KD = 0;                                          % PI -> KD = 0
    KP = cos(phi)/mg;                                % 式 (8-57)
    KI = -ww1*sin(phi)/mg;                           % 由式 (8-58) 且 KD=0
    % 式 (8-52)：積分用梯形法，微分用後向差分
    Dz = KP + (KI*T/2)*tf([1 1],[1 -1],T) + (KD/T)*tf([1 -1],[1 0],T);
    [gm, pm] = my_margin(Dz*Gz, T);
    printf('  %-7.1f %-9.4f %-11.2f %-9.2f %-9.4f %-9.4f %-8.4f %.1f\n', ...
           ww1, mg, pg, phi*180/pi, KP, KI, KI/KP, gm);
    if ww1 == 0.4, Dz_pi = Dz; end
end
[nd, ~] = tfdata(Dz_pi, 'v');
printf('\nww1=0.4 的 D(z) = (%.4fz %+.4f)/(z - 1)\n', nd(1), nd(2));
printf('教材 ww1=0.4：KP=0.4392，KI=0.0040，ww0=0.0092，D(z)=(0.4393z-0.4391)/(z-1)，GM=16.0\n');
printf('教材 ww1=0.3：KP=0.3125，KI=0.0154，ww0=0.0493，D(z)=(0.3129z-0.3121)/(z-1)，GM=18.4\n');
printf('\n=> 設計參數全部與教材相同\n');
printf('注意 phi = -1.32 deg 非常小，正是我們想要的：\n');
printf('     PI 的相位落後不該污染相位裕度頻率附近的相位\n');

### 結果解讀

$$\boxed{D(z)=\frac{0.4393z-0.4391}{z-1}\quad\text{——與教材完全相同}}$$

**$\varphi=-1.32^\circ$ 非常小**——這正是教材強調的：**PI 控制器在相位裕度頻率的相角應該很小**，才不會污染穿越區的相位。

**兩個 $\omega_{w1}$ 的對照**（教材的設計取捨示範）：

| $\omega_{w1}$ | $\omega_{w0}=K_I/K_P$ | 增益裕度 | 教材說的效果 |
|---|---|---|---|
| $0.4$ | $0.0092$ | $16.0$ dB | 上升 2.95 s，超越 13.8%，安定 19.4 s |
| $0.3$ | $0.0493$ | $18.4$ dB | 上升 3.65 s，超越 18.6%，安定 **43.0** s |

> **教材的評語**：$\omega_{w0}=0.0092$ **相當低，系統頻寬被大幅降低。雖然穩態誤差響應很好，但要達到它需要相當長的時間。**

---


## 七、係數量化問題 (8.4 節的警告)

### 教材的警告

> **相位落後濾波器在實作時可能出現數值問題。** 範例 8.1 需要分母係數 $0.999300$，但用有限字長的定點運算可能只能表示成 $0.99609375$——**這個誤差足以大幅改變濾波器特性**。
>
> **原因**：相位落後濾波器的**極點與零點幾乎重合**，位置非常關鍵。
>
> **對照**：相位超前濾波器的極零點**分得很開**，小幅偏移影響很小。

**下一格用數值把「幾乎重合」與「分得很開」量化出來，並實測量化後裕度變多少。**


In [ ]:
%% 實驗 5：係數量化問題（8.4 節的警告）
[nl, dl] = tfdata(Dz_lag,  'v');
[ne, de] = tfdata(Dz_lead, 'v');
printf('相位落後 D(z)：零點 %.4f，極點 %.4f，間距 %.4f  <- 幾乎重合\n', ...
       -nl(2)/nl(1), -dl(2), abs(-nl(2)/nl(1) - (-dl(2))));
printf('相位超前 D(z)：零點 %.4f，極點 %.4f，間距 %.4f  <- 分得很開\n', ...
       -ne(2)/ne(1), -de(2), abs(-ne(2)/ne(1) - (-de(2))));
printf('間距相差 %.0f 倍\n\n', abs(-ne(2)/ne(1)-(-de(2)))/abs(-nl(2)/nl(1)-(-dl(2))));

% 教材的例子：分母係數 0.999300 被量化成 0.99609375
printf('教材的例子：落後濾波器分母係數 0.999300 -> 0.99609375\n');
Dz_q = nl(1)*tf([1 nl(2)/nl(1)], [1 -0.99609375], T);
[g0, p0] = my_margin(Dz_lag*Gz, T);
[gq, pq] = my_margin(Dz_q*Gz, T);
printf('  量化前：GM = %.2f dB，PM = %.2f deg\n', g0, p0);
printf('  量化後：GM = %.2f dB，PM = %.2f deg\n', gq, pq);
printf('  相位裕度變動 %.1f deg\n\n', pq - p0);

% 對照：把超前濾波器的極點做同等比例的擾動
printf('對照：把超前濾波器的極點做「同樣大小」的絕對擾動\n');
dp = 0.9993 - 0.99609375;
Dz_lead_q = ne(1)*tf([1 ne(2)/ne(1)], [1 -(0.8111 - dp)], T);
[gl, pl] = my_margin(Dz_lead*Gz, T);
[glq, plq] = my_margin(Dz_lead_q*Gz, T);
printf('  擾動前：GM = %.2f dB，PM = %.2f deg\n', gl, pl);
printf('  擾動後：GM = %.2f dB，PM = %.2f deg\n', glq, plq);
printf('  相位裕度變動 %.1f deg  <- 遠小於落後濾波器\n', plq - pl);

### 結果解讀

| 濾波器 | 零點 | 極點 | 間距 |
|---|---|---|---|
| **相位落後** | $0.9982$ | $0.9993$ | $\mathbf{0.0011}$ |
| **相位超前** | $0.9711$ | $0.8111$ | $\mathbf{0.1600}$ |

**間距相差約 145 倍。**

**同樣大小的係數擾動**（$0.0032$），對兩種濾波器的影響完全不同——落後濾波器的相位裕度變動遠大於超前濾波器。

> **💡 實務意義**：
> - **定點運算的嵌入式系統**（例如 8/16 位元微控制器）實作相位落後濾波器時，**必須特別注意係數字長**
> - 若字長不夠，設計出來的濾波器在硬體上**不會是你設計的那個**
> - **相位超前濾波器對量化寬容得多**——這是選用它的一個額外理由

---


## 八、五種設計方法的完整比較 (教材 Table 8-3)

**所有設計都用同一個受控體、同樣的目標相位裕度 $55^\circ$。**

| 範例 | 型式 | $D(z)$ |
|---|---|---|
| 8.1 | 落後 | $\dfrac{0.389(z-0.9982)}{z-0.9993}$ |
| 8.2 | 超前 | $\dfrac{6.528(z-0.9711)}{z-0.8111}$ |
| 8.3 | 落後–超前 | $\dfrac{5.227(z-0.9982)(z-0.9792)}{(z-0.9993)(z-0.8604)}$ |
| 8.4 | PI | $\dfrac{0.4393(z-0.99954)}{z-1}$ |
| 8.5 | PID | $\dfrac{28.52(z-0.99986)(z-0.9504)}{z(z-1)}$ |

### ⚠️ `stepinfo` 在 Octave 中不存在

教材例 8.4 的程式用 `stepinfo(Cz)` 取時域指標。**Octave 的 control 4.2.1 沒有這個函數**，本 Notebook 改用手動計算：

```matlab
[y, t] = step(Tcl, 0:T:40);
yf   = y(end);                                  % 終值
i10  = find(y >= 0.1*yf, 1);                    % 10% 點
i90  = find(y >= 0.9*yf, 1);                    % 90% 點
tr   = t(i90) - t(i10);                         % 上升時間
os   = (max(y) - yf)/yf*100;                    % 超越量 (%)
is   = find(abs(y-yf) > 0.02*yf, 1, 'last');    % 最後一次離開 2% 帶
ts   = t(is);                                   % 安定時間
```


In [ ]:
%% 實驗 6：五種設計的完整比較（教材 Table 8-3）
Dz_ll  = 5.227*tf(conv([1 -0.9982],[1 -0.9792]), conv([1 -0.9993],[1 -0.8604]), T);
Dz_pid = 28.52*tf(conv([1 -0.99986],[1 -0.9504]), conv([1 0],[1 -1]), T);
names  = {'8.1 落後', '8.2 超前', '8.3 落後-超前', '8.4 PI', '8.5 PID'};
Ds     = {Dz_lag, Dz_lead, Dz_ll, Dz_pi, Dz_pid};
bkGM   = [16.8 12.4 11.2 16.0 23.3];
bkTr   = [3.15 1.05 1.05 2.95 1.05];
bkTs   = [32.4 5.2 5.3 19.4 3.6];
bkOS   = [14.1 11.4 10.7 13.8 13.0];

printf('  %-14s %-8s %-7s %-8s %-7s %-8s %-7s %-8s %s\n', ...
       '設計', 'GM(dB)', '教材', '上升(s)', '教材', '安定(s)', '教材', '超越(%)', '教材');
figure('Position', [50 50 900 480]);
cols = {'b','r','g','m','k'};
for i = 1:numel(Ds)
    L   = Ds{i}*Gz;
    Tcl = feedback(L, 1);
    [gm, pm] = my_margin(L, T);
    [y, t]   = step(Tcl, 0:T:60);
    yf  = y(end);
    i10 = find(y >= 0.1*yf, 1);  i90 = find(y >= 0.9*yf, 1);
    is  = find(abs(y-yf) > 0.02*yf, 1, 'last');
    printf('  %-14s %-8.1f %-7.1f %-8.2f %-7.2f %-8.1f %-7.1f %-8.1f %.1f\n', ...
           names{i}, gm, bkGM(i), t(i90)-t(i10), bkTr(i), t(is), bkTs(i), ...
           (max(y)-yf)/yf*100, bkOS(i));
    stairs(t, y, cols{i}, 'LineWidth', 1.6); hold on;
end
plot([0 60], [1 1], 'k:', 'LineWidth', 1);
grid on; xlim([0 20]);
title('Table 8-3：五種數位控制器設計的單位階躍響應');
xlabel('時間 t (秒)'); ylabel('c(kT)');
legend([names, {'目標值 1'}], 'Location', 'southeast');

### 結果解讀：本章最重要的一頁

**五種設計的增益裕度全部與教材吻合。**

| 設計 | 優點 | 缺點 |
|---|---|---|
| **落後** | 穩定裕度好（16.8 dB） | **最慢**（安定 32.4 s） |
| **超前** | **反應快**（上升 1.05 s） | 斜坡誤差沒改善（1.00） |
| **落後–超前** | 快 **且** 斜坡誤差降到 0.50 | 增益裕度較低（11.2 dB） |
| **PI** | **斜坡誤差 0** | 頻寬低，安定 19.4 s |
| **PID** | **全面最佳**：快（1.05）、穩（23.3 dB）、誤差 0 | 三個參數要調 |

$$\boxed{\text{PID}=\underbrace{\text{積分項}}_{\text{落後的效果}}+\underbrace{\text{微分項}}_{\text{超前的效果}}\ \Longrightarrow\ \text{同時拿到兩邊的好處}}$$

**從階躍響應圖上看**：藍色（落後）與洋紅（PI）明顯拖得很長；紅、綠、黑（超前、落後–超前、PID）幾乎重疊在最快的那一組。

> **💡 但這不代表 PID 永遠最好**。教材導論已經提醒過：**所有數值設計都基於不精確的模型，最終要靠實體實驗迭代**。PID 的三個參數也意味著三倍的調校工作量，而且**微分項會放大高頻雜訊**（所以式 8-50 要加極點）。

---


## 九、本章總結

### 公式速查表

| 式號 | 公式 | 用途 | 對應實驗 |
|---|---|---|---|
| (8-13) | $D(w)=a_0\dfrac{1+w/\omega_{w0}}{1+w/\omega_{wp}}$ | **補償器標準形式** | 2, 3 |
| — | $\omega_{w0}<\omega_{wp}$→超前；$>$→落後 | **分類** | 2, 3 |
| (8-14)(8-15) | $K_d,z_0,z_p$ | **$D(w)\to D(z)$（最後一步）** | 2, 3 |
| (8-20)(8-21) | $\omega_{w0}=0.1\omega_{w1}$，$\omega_{wp}=\frac{0.1\omega_{w1}}{a_0| G|}$ | **落後設計** | 2 |
| (8-32)(8-33) | $\varphi$、$a_1$、$b_1$ | **超前設計** | 3 |
| (8-45) | $D(w)=K_P+\frac{K_I}{w}+K_Dw$ | **PID** | 4 |
| (8-52) | PID 的離散化 | 積分用梯形、微分用後向差分 | 4 |
| (8-56)(8-57)(8-58) | $\varphi$、$K_P$、$K_D/K_I$ 關係 | **PID 設計** | 4 |

### ⚠️ Octave 相容性總表

| 教材用的 | Octave 狀況 | 替代方案 |
|---|---|---|
| **`bode(Gz,w)` 取相位** | **含 $z=1$ 極點時錯 $90\sim180^\circ$** | **`freqresp(Gz,w)`** |
| `margin(L)` | 常回傳 `pm=180`、`wp=NaN` | 自寫 `my_margin` 掃頻 |
| `stepinfo(Cz)` | 不存在 | 手動由 `step` 輸出計算 |
| `pidtool` | 不存在 | 手動實作式 (8-56)~(8-58) |
| `sisotool` | 不存在 | 手動實作 / 用 Table 8-3 驗證 |
| `angle()` | 回傳 $(-180^\circ,180^\circ]$ | 低於 $-180^\circ$ 時手動減 360 |
| 腳本內函數位置 | Octave 要先定義、MATLAB 要在檔尾 | **拆成獨立的函數檔** |

### 承先啟後

本章完成了**古典頻率響應設計**的完整流程：

$$\text{規格}\ \to\ \text{選 }\omega_{w1}\ \to\ \text{設計方程式}\ \to\ D(w)\ \to\ \text{式 (8-14)}\ \to\ D(z)\ \to\ \text{檢查裕度}$$

**但這套方法有兩個本質限制**（教材導論已經預告）：

1. **本質上是試誤法**——$\omega_{w1}$ 要試、$K_D$ 與 $K_I$ 的分配要試
2. **只能間接控制閉迴路極點**——你指定的是「相位裕度」，不是極點位置

**第 9 章的極點安置法直接解決第二點**：

> **直接指定閉迴路極點要落在哪裡，用阿克曼公式一次反解出增益。**

完整範例見 [`course/ackermann`](../ackermann/full-Ackermann-formula-example.md)——那裡的「向系統下訂單」就是本章「選 $\omega_{w1}$ 與 $\eta_m$」的升級版，而且**不需要試誤**。
